In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

In [2]:
import sklearn
np.__version__, pd.__version__, sklearn.__version__

('2.4.3', '3.0.1', '1.8.0')

### 전처리

In [ ]:
df = pd.read_csv("set_06_data.csv")
df.head(2)

In [ ]:
# 단계 1)
df_base = df.drop(columns = ["city", "company_size", "company_type"])
df_base.head(1)

In [ ]:
# 단계 2)
# df_base.isna().sum()
pd.DataFrame(dict(dtype = df_base.dtypes, 
                  na_cnt = df_base.isna().sum()))

In [ ]:
df_base = df_base.dropna()
len(df_base)

In [8]:
# 단계 3)
df_base = df_base.loc[~df_base["exp"].isin([">20", "<1"]), ]
df_base["exp"] = df_base["exp"].astype("int")

In [9]:
# 단계 4)
df_base = df_base.loc[~df_base["last_new_job"].isin([">4", "never"]), ]
df_base["last_new_job"] = df_base["last_new_job"].astype("int")

In [11]:
df_base = df_base.reset_index(drop = True)

In [12]:
len(df_base)

7522

### Q1.

In [ ]:
df_base["edu_level"].unique()

In [ ]:
df_q1 = df_base.loc[df_base["edu_level"].isin(["Masters", "Phd"]), ["relevant_exp", "target"]]
df_q1.head(3)

In [16]:
df_A = df_q1.loc[df_q1["relevant_exp"] == "No", ]
df_B = df_q1.loc[df_q1["relevant_exp"] == "Has", ]
len(df_A), len(df_B)

(366, 1686)

In [20]:
stat_A = df_A["target"].value_counts(normalize = True)[1]
stat_B = df_B["target"].value_counts(normalize = True)[1]
stat_A, stat_B

(np.float64(0.2814207650273224), np.float64(0.17734282325029657))

In [22]:
round(stat_A / stat_B, 2)

np.float64(1.59)

### Q2.

In [23]:
df_base.columns

Index(['enrollee_id', 'city_development_index', 'gender', 'relevant_exp',
       'enrolled_university', 'edu_level', 'major_discipline', 'exp',
       'last_new_job', 'training_hours', 'target', 'Xgrp'],
      dtype='str')

In [26]:
# 단계 1)
ls_cols = ["gender", "relevant_exp", "enrolled_university", "edu_level", "major_discipline"]
# df_q2_dum = pd.get_dummies(df_base[ls_cols], columns = ls_cols) # 시험버전
df_q2_dum = pd.get_dummies(df_base[ls_cols], columns = ls_cols, dtype = "int") # 시험버전
df_q2_dum.head(1)

,gender_Female,gender_Male,gender_Other,relevant_exp_Has,relevant_exp_No,enrolled_university_Full time course,enrolled_university_Part time course,enrolled_university_no_enrollment,edu_level_Graduate,edu_level_Masters,edu_level_Phd,major_discipline_Arts,major_discipline_Business Degree,major_discipline_Humanities,major_discipline_No Major,major_discipline_Other,major_discipline_STEM
0,0,1,0,1,0,0,0,1,1,0,0,0,0,0,0,0,1


In [ ]:
# 단계 2)
# df_q2_dum.head(1).T.reset_index()
df_q2_dum.head(1).transpose().reset_index()

In [35]:
df_q2_dum.columns[[2, 4, 7, 10, 16]]

Index(['gender_Other', 'relevant_exp_No', 'enrolled_university_no_enrollment',
       'edu_level_Phd', 'major_discipline_STEM'],
      dtype='str')

In [ ]:
df_q2_dum2 = df_q2_dum.drop(columns = df_q2_dum.columns[[2, 4, 7, 10, 16]])
df_q2_dum2.head(2)

In [ ]:
# 단계 3)
ls_col_y = ["target", "Xgrp"]
ls_col_x = ["city_development_index", "exp", "last_new_job", "training_hours"]
df_job2 = pd.concat([df_base[ls_col_y + ls_col_x], df_q2_dum2], axis = 1)
df_job2.head(2)

In [38]:
# concat() 사용 시 주의점!! 🚨🚨🚨
# index가 어긋나서 결측치가 생길 수 있음!
df_job2.isna().sum()

target                                  0
Xgrp                                    0
city_development_index                  0
exp                                     0
last_new_job                            0
training_hours                          0
gender_Female                           0
gender_Male                             0
relevant_exp_Has                        0
enrolled_university_Full time course    0
enrolled_university_Part time course    0
edu_level_Graduate                      0
edu_level_Masters                       0
major_discipline_Arts                   0
major_discipline_Business Degree        0
major_discipline_Humanities             0
major_discipline_No Major               0
major_discipline_Other                  0
dtype: int64

In [ ]:
model_lr = LogisticRegression(fit_intercept = True, random_state = 1234,
                              C = 100000, max_iter = 1000, solver = "liblinear")
model_lr.fit(X = df_job2.drop(columns = ["target", "Xgrp"]),
             y = df_job2["target"])

In [ ]:
model_lr.coef_ # 로지스틱 회귀모델의 회귀계수

$$ OR(Odds Ratio) = e^{\beta_i} $$

In [ ]:
arr_OR = np.exp(model_lr.coef_)
arr_OR

In [46]:
arr_OR.max() # 1.67

np.float64(1.6738948061770285)

### Q3. 

In [49]:
# 단계 1)
df_train = df_job2.loc[df_job2["Xgrp"] == "train", ]
df_test  = df_job2.loc[df_job2["Xgrp"] == "test",  ]
len(df_train), len(df_test)

(4706, 2816)

In [ ]:
# 단계 2)
df_train_X = df_train.drop(columns = ["target", "Xgrp"])
df_train_y = df_train["target"]

In [51]:
# 단계 3)
model_knn = KNeighborsClassifier(n_neighbors = 3)
model_knn.fit(X = df_train.drop(columns = ["target", "Xgrp"]),
              y = df_train["target"])
pred_knn = model_knn.predict(df_test.drop(columns = ["target", "Xgrp"]))
pred_knn[:5]

array([0, 0, 0, 1, 0])

In [54]:
# 단계 4)
from sklearn.metrics import accuracy_score
round(accuracy_score(y_true = df_test["target"], y_pred = pred_knn), 2)

0.69

In [ ]:
model = LogisticRegression(random_state = 123, max_iter = 1000) # max_iter는 몰라도 됨. 시험범위 아님.
model.fit(X = df_train.drop(columns = ["target", "Xgrp"]),
          y = df_train["target"])

In [58]:
pred_pr = model.predict_proba(df_test.drop(columns = ["target", "Xgrp"]))
pred_pr[:4] # 이진 분류의 경우에는 두 번째 열을 사용해야 함.

array([[0.82549265, 0.17450735],
       [0.73186109, 0.26813891],
       [0.92400979, 0.07599021],
       [0.85230264, 0.14769736]])

In [ ]:
pred_c1 = np.where(pred_pr[:, 1] > 0.5, 1, 0) # .predict() 를 사용한 것과 같음.
pred_c1[:4]

In [60]:
pred_c2 = np.where(pred_pr[:, 1] > 0.3, 1, 0)
pred_c2[:4]

array([0, 0, 0, 0])

In [62]:
from sklearn.metrics import f1_score
f1_score(y_true = df_test["target"], y_pred = pred_c1)

0.29959514170040485

In [63]:
f1_score(y_true = df_test["target"], y_pred = pred_c2)

0.5282051282051282

In [64]:
from sklearn.metrics import roc_auc_score

In [65]:
roc_auc_score(y_true = df_test["target"], y_score = pred_pr[:, 1]) # 확률값 넣어야 함!! 🎯🎯

0.7380573059167785

In [67]:
roc_auc_score(y_true = df_test["target"], y_score = pred_c1) # 분류값을 넣으면 에러도 안나고 값도 틀림. 🚨🚨🚨🚨

0.5744852793594118